In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1:
import os

csv_file_path = os.path.join(path, '/kaggle/input/q1-ka-ai-2026/Q1_data.csv')

df = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Q1_data'].dropna(), bins=30, edgecolor='black', color='yellow')
plt.title('delivery_time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns="Order_ID", axis=1)


In [ ]:
# Task 2: Write your code here:
# Get all categorical columns
cat_cols = df.select_dtypes(include='object').columns

# Replace missing categorical values with 'unknown' so encoding works
df[cat_cols] = df[cat_cols].fillna("unknown")
# Get all numerical columns
num_cols = df.select_dtypes(include='number').columns

# Fill missing numerical values with the mean so the model doesn’t crash
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

In [ ]:
# Task 3: Write your code here:
# Remove duplicated rows to avoid fake high accuracy
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder

In [ ]:
# Task 5: Write your code here:
# Import StandardScaler to scale features
from sklearn.preprocessing import StandardScaler


# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# IMPORTANT: we NEVER scale y

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 2. Use KFold (Regression task)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

# 3. Train RandomForest model across folds
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # 4. Evaluate using MAE ONLY
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

# 5. Print the averaged score across all folds
print(f"Average MAE across all folds: {np.mean(mae_scores):,.2f}")



In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 5))

plt.plot(train_loss, label='Train Loss')
plt.plot(val_loss, label='Validation Loss')
plt.title('Loss over Epochs')

In [ ]:
# Task 2: Write your code here:
 # Make predictions on the test set (hard labels)
    y_pred = model.predict(X_test)

In [ ]:

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# 1. Split into features and target
X = df.drop('delivery_time', axis=1)
y = df['delivery_time']

# 2. Setup KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

print("Starting Ensemble Training (RandomForest + CatBoost)...")

# 3. KFold Loop with 2 models
for fold, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # --- Model 1: RandomForest ---
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)

    # --- Model 2: CatBoost ---
    cb_model = CatBoostRegressor(verbose=0, random_state=42)
    cb_model.fit(X_train, y_train)
    cb_preds = cb_model.predict(X_test)

    # 4. Average the predictions (Ensemble)
    # This combines the "votes" of both models
    averaged_preds = (rf_preds + cb_preds) / 2

    # 5. Calculate MAE for the ensemble
    fold_mae = mean_absolute_error(y_test, averaged_preds)
    ensemble_mae_scores.append(fold_mae)

    print(f"Fold {fold+1} MAE: {fold_mae:.4f}")

# 6. Final Averaged Score
print("-" * 30)
print(f"Final Ensemble Average MAE: {np.mean(ensemble_mae_scores):.4f}")